In [2]:
import joblib
import re
import string
import demoji
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import tkinter as tk

# Download NLTK resources once
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def full_preprocess(text):
    text = demoji.replace_with_desc(text, sep=" ")
    text = text.lower()
    text = re.sub(r'^rt\s+', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens if token.isalpha() and token not in stop_words
    ]
    return ' '.join(tokens)

vectorizer = joblib.load(r'C:\Users\sujal\Projects\Cyber Bullying Detection\cyberbullying_simple_models\tfidf_vectorizer.pkl')
svc_model = joblib.load(r'C:\Users\sujal\Projects\Cyber Bullying Detection\cyberbullying_simple_models\linear_svc_model.pkl')
logreg_model = joblib.load(r'C:\Users\sujal\Projects\Cyber Bullying Detection\cyberbullying_simple_models\logistic_regression_model.pkl')
nb_model = joblib.load(r'C:\Users\sujal\Projects\Cyber Bullying Detection\cyberbullying_simple_models\naive_bayes_model.pkl')

MODELS = [
    ("Linear SVC", svc_model),
    ("Logistic Regression", logreg_model),
    ("Naive Bayes", nb_model)
]

def predict_text(text):
    processed = full_preprocess(text)
    vect = vectorizer.transform([processed])
    results = []
    for name, model in MODELS:
        pred = model.predict(vect)[0]
        label = "Cyberbullying" if pred == 1 else "Not Cyberbullying"
        prob = None
        if hasattr(model, "predict_proba"):
            # Get correct class index in .predict_proba result
            class_index = list(model.classes_).index(pred)
            prob = model.predict_proba(vect)[0][class_index]
        if prob is not None:
            outcome = f"{name}: {label} (Prob: {prob:.2f})"
        else:
            outcome = f"{name}: {label}"
        results.append((pred, outcome))
    return results


def on_send():
    text = input_field.get()
    results = predict_text(text)
    is_cyberbullying = any(pred == 1 for pred, _ in results)
    output = "\n".join(outcome for _, outcome in results)
    if is_cyberbullying:
        warning_label.config(text="Warning! Cyberbullying detected.\n" + output, fg="red")
    else:
        warning_label.config(text="Message sent!\n" + output, fg="green")
        input_field.delete(0, tk.END)

# Tkinter GUI
root = tk.Tk()
root.title("Safe Keyboard (Simple Model)")
input_field = tk.Entry(root, width=50)
input_field.pack(pady=10)
warning_label = tk.Label(root, text="", font=("Arial", 12))
warning_label.pack(pady=5)
send_btn = tk.Button(root, text="Send", command=on_send)
send_btn.pack(pady=10)
root.mainloop()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sujal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sujal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sujal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
